# 📚 Define Libraries

In [2]:
from google.cloud import bigquery
from datetime import datetime

# ---------------------------------------------------------
# 1. Create BigQuery client
# ---------------------------------------------------------
client = bigquery.Client(project="prj-prod-dataplatform")

C:\Users\Dwaipayan\AppData\Roaming\Python\Python312\site-packages\google\auth\_default.py:76: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


# ❄️ beta_stack_model_cash_v1

In [3]:
print("Starting BigQuery job...")

sq = """ 
create or replace table `dap_ds_poweruser_playground.beta_stack_model_cash_v1` as 
select a.* 
, date(lmt.startApplyDateTime) application_date
from 
(
(WITH parsed as (
select customerId, digitalLoanAccountId,modelDisplayName,modelVersionId,start_time,end_time,prediction,trenchCategory,
REPLACE(REPLACE(calcFeature, "'", '"'), "None", "null") AS calcFeatures,
REPLACE(REPLACE(prediction, "'", '"'), "None", "null") AS prediction_clean,
 deviceOs osType,
FROM `prj-prod-dataplatform.audit_balance.ml_model_run_details`
where modelDisplayName in  ('Beta-Cash-Stack-Model', 'beta_stack_model_cash')
and modelVersionId = 'v1'
)
,
latest_request as (
select * from parsed
QUALIFY ROW_NUMBER() OVER (PARTITION BY customerId, digitalLoanAccountId,modelDisplayName ORDER BY start_time DESC ) = 1),
modelname as (
  select 
  safe_cast(p.customerId as string) customerId,
  p.digitalLoanAccountId,
  safe_cast(prediction as numeric) prediction,
 calcFeatures,
  p.trenchCategory,
  case when modelDisplayName like 'Beta-Cash-Stack-Model' then 'beta_stack_model_cash' else p.modelDisplayName end as modelDisplayName,
  p.modelVersionId, osType
  from latest_request p
  -- left join model_run m on m.digitalLoanAccountId = p.digitalLoanAccountId
  )
select
t.* EXCEPT(calcFeatures),
SAFE_CAST(JSON_VALUE(calcFeatures, '$.demo_score') AS FLOAT64) AS demo_score,
SAFE_CAST(JSON_VALUE(calcFeatures, '$.credo_score') AS FLOAT64) AS credo_score,
SAFE_CAST(JSON_VALUE(calcFeatures, '$.trx_score') AS FLOAT64) AS trx_score,
SAFE_CAST(JSON_VALUE(calcFeatures, '$.app_score') AS FLOAT64) AS app_score,
'Prod' DataSelection
from modelname t
)
union all 
(
WITH 
parsed as (
select customerId, digitalLoanAccountId,modelDisplayName,modelVersionId,start_time,end_time,prediction,trenchCategory,
REPLACE(REPLACE(calcFeature, "'", '"'), "None", "null") AS calcFeatures,
Data_selection,  deviceOs osType,
FROM prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116
where modelDisplayName in  ('Beta-Cash-Stack-Model', 'beta_stack_model_cash')
and modelVersionId = 'v1'
),
  modelname as (
  select 
  safe_cast(customerId as string) customerId,
  digitalLoanAccountId,
  safe_cast(prediction as numeric) prediction,
  calcFeatures,
  trenchCategory,
  case when modelDisplayName like 'Beta-Cash-Stack-Model' then 'beta_stack_model_cash' else modelDisplayName end as    modelDisplayName,
  modelVersionId,
  Data_selection, osType
  from parsed
  )
select
t.* EXCEPT(calcFeatures, Data_selection),
SAFE_CAST(JSON_VALUE(calcFeatures, '$.demo_score') AS FLOAT64) AS demo_score,
SAFE_CAST(JSON_VALUE(calcFeatures, '$.credo_score') AS FLOAT64) AS credo_score,
SAFE_CAST(JSON_VALUE(calcFeatures, '$.trx_score') AS FLOAT64) AS trx_score,
SAFE_CAST(JSON_VALUE(calcFeatures, '$.app_score') AS FLOAT64) AS app_score,
Data_selection DataSelection
from modelname t
)
) a 
left join (select digitalLoanAccountId,startApplyDateTime from `risk_credit_mis.loan_master_table`) lmt on lmt.digitalLoanAccountId = a.digitalLoanAccountId
"""

query_job = client.query(sq)
# Wait until query finishes
query_job.result()
print("BigQuery job completed successfully!")


# ---------------------------------------------------------
# 3. Job details
# ---------------------------------------------------------
print("\n========== JOB DETAILS ==========")

print(f"Job ID              : {query_job.job_id}")
print(f"State               : {query_job.state}")
print(f"Created             : {query_job.created}")
print(f"Started             : {query_job.started}")
print(f"Ended               : {query_job.ended}")

# Bytes processed
if query_job.total_bytes_processed is not None:
    print(
        f"Bytes processed     : "
        f"{query_job.total_bytes_processed:,} "
        f"({query_job.total_bytes_processed / 1024**3:.3f} GB)"
    )

if query_job.total_bytes_billed is not None:
    print(
        f"Bytes billed        : "
        f"{query_job.total_bytes_billed:,} "
        f"({query_job.total_bytes_billed / 1024**3:.3f} GB)"
    )


# ---------------------------------------------------------
# 4. Get newly created table
# ---------------------------------------------------------
table_id = (
    "prj-prod-dataplatform."
    "dap_ds_poweruser_playground."
    "beta_stack_model_cash_v1"
)

table = client.get_table(table_id)


# ---------------------------------------------------------
# 5. Table shape and metadata
# ---------------------------------------------------------
rows = table.num_rows
columns = len(table.schema)

print("\n========== TABLE DETAILS ==========")

print(f"Table               : {table_id}")
print(f"Shape               : ({rows:,}, {columns})")
print(f"Rows                : {rows:,}")
print(f"Columns             : {columns:,}")

print(
    f"Table size          : "
    f"{table.num_bytes:,} bytes "
    f"({table.num_bytes / 1024**2:.2f} MB)"
)

print(f"Created             : {table.created}")
print(f"Last modified       : {table.modified}")


# ---------------------------------------------------------
# 6. Print schema / column details
# ---------------------------------------------------------
print("\n========== SCHEMA ==========")

for i, field in enumerate(table.schema, start=1):
    print(
        f"{i:02d}. "
        f"{field.name:<30} "
        f"{field.field_type:<15} "
        f"{field.mode}"
    )

Starting BigQuery job...
BigQuery job completed successfully!

========== JOB DETAILS ==========
Job ID              : a1e5aab7-2353-4275-a7db-169ca5bd0595
State               : DONE
Created             : 2026-10-06 03:53:57.005000+00:00
Started             : 2026-10-06 03:53:57.326000+00:00
Ended               : 2026-10-06 03:54:03.712000+00:00
Bytes processed     : 10,838,538,086 (10.094 GB)
Bytes billed        : 10,839,130,112 (10.095 GB)

========== TABLE DETAILS ==========
Table               : prj-prod-dataplatform.dap_ds_poweruser_playground.beta_stack_model_cash_v1
Shape               : (640,806, 13)
Rows                : 640,806
Columns             : 13
Table size          : 92,043,091 bytes (87.78 MB)
Created             : 2026-10-06 03:54:03.565000+00:00
Last modified       : 2026-10-06 03:54:03.566000+00:00

========== SCHEMA ==========
01. customerId                     STRING          NULLABLE
02. digitalLoanAccountId           STRING          NULLABLE
03. prediction     